In [23]:
import textwrap, runpy
from pathlib import Path

In [24]:
import re
import xarray as xr
import rioxarray as rxr
from pathlib import Path

In [25]:
print("Starting NBR→NetCDF...")

Starting NBR→NetCDF...


In [27]:
# 1) Input folder
in_dir = Path(r"D:\Farhan\NBR Data")
files = sorted(in_dir.glob("*.tif"))
print(f"Found {len(files)} GeoTIFF(s) in {in_dir}")
if not files:
    raise FileNotFoundError(f"No .tif files found in {in_dir}")

Found 23 GeoTIFF(s) in D:\Farhan\NBR Data


In [28]:
# 2) Sort files by year if present
def year_key(p):
    m = re.search(r"(19|20)\d{2}", p.stem)
    return int(m.group(0)) if m else p.name

files = sorted(files, key=year_key)
for idx, f in enumerate(files, 1):
    print(f"[{idx}/{len(files)}] {f.name}")

rasters = []
ref = None

[1/23] NBR_2002.tif
[2/23] NBR_2003.tif
[3/23] NBR_2004.tif
[4/23] NBR_2005.tif
[5/23] NBR_2006.tif
[6/23] NBR_2007.tif
[7/23] NBR_2008.tif
[8/23] NBR_2009.tif
[9/23] NBR_2010.tif
[10/23] NBR_2011.tif
[11/23] NBR_2012.tif
[12/23] NBR_2013.tif
[13/23] NBR_2014.tif
[14/23] NBR_2015.tif
[15/23] NBR_2017.tif
[16/23] NBR_2018-DA142189.tif
[17/23] NBR_2018.tif
[18/23] NBR_2019.tif
[19/23] NBR_2020.tif
[20/23] NBR_2021.tif
[21/23] NBR_2022.tif
[22/23] NBR_2023.tif
[23/23] NBR_2024.tif


In [39]:
import re
from pathlib import Path
import xarray as xr
import rioxarray as rxr

# --- SETTINGS ---
IN_DIR = Path(r"D:\Farhan\NBR Data")
OUT_NC = r"D:\Farhan\NBR_Data_Timeseries.nc"

print("Scanning:", IN_DIR)
files = sorted(IN_DIR.glob("*.tif"))

if not files:
    raise FileNotFoundError(f"No .tif files found in {IN_DIR}")

def year_key(p):
    m = re.search(r"(19|20)\d{2}", p.stem)
    return int(m.group(0)) if m else p.name

files = sorted(files, key=year_key)
print(f"Found {len(files)} GeoTIFFs")
for i, f in enumerate(files, 1):
    print(f"[{i}/{len(files)}] {f.name}")

rasters = []
ref = None

for i, f in enumerate(files):
    da_i = rxr.open_rasterio(f, masked=True)  # no dask/chunks to keep it simple
    if "band" in da_i.dims and da_i.sizes.get("band", 1) == 1:
        da_i = da_i.squeeze("band", drop=True)

    if ref is None:
        ref = da_i
    else:
        same_grid = (
            (da_i.rio.crs == ref.rio.crs) and
            (da_i.rio.transform() == ref.rio.transform()) and
            (da_i.rio.width == ref.rio.width) and
            (da_i.rio.height == ref.rio.height)
        )
        if not same_grid:
            da_i = da_i.rio.reproject_match(ref)

    m = re.search(r"(19|20)\d{2}", f.stem)
    time_val = int(m.group(0)) if m else i
    da_i = da_i.expand_dims("time").assign_coords(time=[time_val])

    rasters.append(da_i)

stack = xr.concat(rasters, dim="time")
stack.name = "NBR"
stack.attrs.update({
    "long_name": "Normalized Burn Ratio",
    "units": "1",
    "crs": ref.rio.crs.to_string() if ref.rio.crs else "",
})

encoding = {"NBR": {"zlib": True, "complevel": 4, "dtype": "float32", "_FillValue": -9999.0}}

print("Writing NetCDF:", OUT_NC)
stack.to_netcdf(OUT_NC, engine="netcdf4", encoding=encoding)
print("Done.")


Scanning: D:\Farhan\NBR Data
Found 23 GeoTIFFs
[1/23] NBR_2002.tif
[2/23] NBR_2003.tif
[3/23] NBR_2004.tif
[4/23] NBR_2005.tif
[5/23] NBR_2006.tif
[6/23] NBR_2007.tif
[7/23] NBR_2008.tif
[8/23] NBR_2009.tif
[9/23] NBR_2010.tif
[10/23] NBR_2011.tif
[11/23] NBR_2012.tif
[12/23] NBR_2013.tif
[13/23] NBR_2014.tif
[14/23] NBR_2015.tif
[15/23] NBR_2017.tif
[16/23] NBR_2018-DA142189.tif
[17/23] NBR_2018.tif
[18/23] NBR_2019.tif
[19/23] NBR_2020.tif
[20/23] NBR_2021.tif
[21/23] NBR_2022.tif
[22/23] NBR_2023.tif
[23/23] NBR_2024.tif
Writing NetCDF: D:\Farhan\NBR_Data_Timeseries.nc
Done.


In [40]:
import re
import numpy as np
from pathlib import Path
import xarray as xr
import rioxarray as rxr

IN_DIR = Path(r"D:\Farhan\NBR Data")
OUT_NC = r"D:\Farhan\NBR_Data_Timeseries.nc"

files = sorted(IN_DIR.glob("*.tif"))
if not files:
    raise FileNotFoundError(f"No .tif files found in {IN_DIR}")

def year_key(p):
    m = re.search(r"(19|20)\d{2}", p.stem)
    return int(m.group(0)) if m else p.name

files = sorted(files, key=year_key)

rasters = []
ref = None

for i, f in enumerate(files):
    da_i = rxr.open_rasterio(f, masked=True)
    if "band" in da_i.dims and da_i.sizes.get("band", 1) == 1:
        da_i = da_i.squeeze("band", drop=True)

    if ref is None:
        ref = da_i
    else:
        same_grid = (
            (da_i.rio.crs == ref.rio.crs) and
            (da_i.rio.transform() == ref.rio.transform()) and
            (da_i.rio.width == ref.rio.width) and
            (da_i.rio.height == ref.rio.height)
        )
        if not same_grid:
            da_i = da_i.rio.reproject_match(ref)

    # ---- make CF-compliant time from YEAR ----
    m = re.search(r"(19|20)\d{2}", f.stem)
    year = int(m.group(0)) if m else (2000 + i)     # fallback year
    t = np.datetime64(f"{year}-01-01")              # e.g., 2018-01-01

    # add time dimension first, then its coordinate
    da_i = da_i.expand_dims({"time": [t]})

    rasters.append(da_i)

# stack to (time, y, x)
stack = xr.concat(rasters, dim="time").sortby("time")
stack.name = "NBR"
stack.attrs.update({"long_name": "Normalized Burn Ratio", "units": "1"})

# ---- ensure CRS/grid mapping is written so QGIS can geolocate it ----
# (adds a 'spatial_ref' coord and sets grid_mapping)
stack.rio.write_crs(ref.rio.crs, inplace=True)
stack.attrs["grid_mapping"] = "spatial_ref"

# compression + explicit time encoding for CF
encoding = {
    "NBR": {"zlib": True, "complevel": 4, "dtype": "float32", "_FillValue": -9999.0},
    "time": {"units": "days since 1970-01-01", "calendar": "standard"},
}

stack.to_netcdf(OUT_NC, engine="netcdf4", encoding=encoding)
print("✅ Wrote:", OUT_NC, "| shape:", stack.shape, "| dims:", stack.dims)
print("CRS:", stack.rio.crs)

✅ Wrote: D:\Farhan\NBR_Data_Timeseries.nc | shape: (23, 9627, 5680) | dims: ('time', 'y', 'x')
CRS: EPSG:4326
